In [8]:
import numpy as np

def generate_arduino_data(embeddings):
    """Convert UMAP embeddings to Arduino array format"""
    print("\nEmbeddings shape:", embeddings.shape)
    print("First few embeddings points:", embeddings[:5])
    
    # Normalize UMAP coordinates to -128 to 127 range (for int8_t)
    x_normalized = (embeddings[:, 0] - embeddings[:, 0].min()) / (embeddings[:, 0].max() - embeddings[:, 0].min()) * 255 - 128
    y_normalized = (embeddings[:, 1] - embeddings[:, 1].min()) / (embeddings[:, 1].max() - embeddings[:, 1].min()) * 255 - 128
    
    # Generate Arduino code
    arduino_code = "const ColorPoint umapData[] = {\n"
    
    # Generate data points - using normalized coordinates for RGB values
    for i in range(min(64, len(embeddings))):
        # Map x,y coordinates to RGB values (just for visualization)
        r = int(((x_normalized[i] + 128) / 255) * 255)
        g = int(((y_normalized[i] + 128) / 255) * 255)
        b = 127  # constant blue value for contrast
        
        arduino_code += f"  {{{r}, {g}, {b}, {int(x_normalized[i])}, {int(y_normalized[i])}}},"
        arduino_code += f"  // Point {i}\n"
    
    arduino_code += "};\n"
    
    return arduino_code

try:
    # Load the embeddings from .npy file
    embeddings = np.load('umap_embeddings.npy')
    print("Loaded embeddings array with shape:", embeddings.shape)
    
    # Generate and save Arduino code
    arduino_code = generate_arduino_data(embeddings)
    
    # Save as header file
    with open("umap_data.h", "w") as f:
        f.write("// Generated UMAP data for Arduino\n")
        f.write("#ifndef UMAP_DATA_H\n")
        f.write("#define UMAP_DATA_H\n\n")
        f.write("struct ColorPoint {\n")
        f.write("  uint8_t r;\n")
        f.write("  uint8_t g;\n")
        f.write("  uint8_t b;\n")
        f.write("  int8_t x;\n")
        f.write("  int8_t y;\n")
        f.write("};\n\n")
        f.write(arduino_code)
        f.write("\n#endif // UMAP_DATA_H\n")
    
    print(f"\nArduino data saved to umap_data.h")
    print(f"Number of points: {min(64, len(embeddings))}")

except Exception as e:
    print(f"Error: {str(e)}")
    print("\nPlease check your NPY file and try again.")

Loaded embeddings array with shape: (9882, 2)

Embeddings shape: (9882, 2)
First few embeddings points: [[-1.6363834  5.4414186]
 [-1.6129396  5.681586 ]
 [-1.6703343  5.8153462]
 [-1.6753109  6.0149055]
 [-1.6232986  6.335668 ]]

Arduino data saved to umap_data.h
Number of points: 64


In [9]:
import numpy as np

def generate_arduino_data(embeddings):
    """Convert UMAP embeddings to Arduino array format with better color mapping"""
    print("\nEmbeddings shape:", embeddings.shape)
    
    # Normalize coordinates to -128 to 127 range for position
    x_normalized = (embeddings[:, 0] - embeddings[:, 0].min()) / (embeddings[:, 0].max() - embeddings[:, 0].min()) * 255 - 128
    y_normalized = (embeddings[:, 1] - embeddings[:, 1].min()) / (embeddings[:, 1].max() - embeddings[:, 1].min()) * 255 - 128
    
    # Create more distinctive color mapping
    # Map x coordinate to red-green spectrum
    r = (x_normalized + 128) / 255.0
    g = (y_normalized + 128) / 255.0
    
    # Add blue based on distance from center
    b = np.sqrt(x_normalized**2 + y_normalized**2) / np.sqrt(128**2 + 128**2)
    
    # Generate Arduino code
    arduino_code = "const ColorPoint umapData[] = {\n"
    
    for i in range(min(64, len(embeddings))):
        # Scale colors to full range and create more contrast
        red = int(max(0, min(255, r[i] * 255)))
        green = int(max(0, min(255, g[i] * 255)))
        blue = int(max(0, min(255, b[i] * 255)))
        
        # If colors are too similar, increase contrast
        if abs(red - green) < 30 and abs(green - blue) < 30:
            max_channel = max(red, green, blue)
            if max_channel == red:
                red = min(255, red * 1.5)
                green = max(0, green * 0.7)
                blue = max(0, blue * 0.7)
            elif max_channel == green:
                green = min(255, green * 1.5)
                red = max(0, red * 0.7)
                blue = max(0, blue * 0.7)
            else:
                blue = min(255, blue * 1.5)
                red = max(0, red * 0.7)
                green = max(0, green * 0.7)
        
        arduino_code += f"  {{{int(red)}, {int(green)}, {int(blue)}, {int(x_normalized[i])}, {int(y_normalized[i])}}},"
        arduino_code += f"  // Point {i}\n"
    
    arduino_code += "};\n"
    return arduino_code

try:
    # Load the embeddings from .npy file
    embeddings = np.load('umap_embeddings.npy')
    print("Loaded embeddings array with shape:", embeddings.shape)
    
    # Generate and save Arduino code
    arduino_code = generate_arduino_data(embeddings)
    
    # Save as header file
    with open("umap_data2.h", "w") as f:
        f.write("// Generated UMAP data for Arduino\n")
        f.write("#ifndef UMAP_DATA_H\n")
        f.write("#define UMAP_DATA_H\n\n")
        f.write("struct ColorPoint {\n")
        f.write("  uint8_t r;\n")
        f.write("  uint8_t g;\n")
        f.write("  uint8_t b;\n")
        f.write("  int8_t x;\n")
        f.write("  int8_t y;\n")
        f.write("};\n\n")
        f.write(arduino_code)
        f.write("\n#endif // UMAP_DATA_H\n")
    
    print(f"\nArduino data saved to umap_data.h")
    print(f"Number of points: {min(64, len(embeddings))}")

except Exception as e:
    print(f"Error: {str(e)}")
    print("\nPlease check your NPY file and try again.")

Loaded embeddings array with shape: (9882, 2)

Embeddings shape: (9882, 2)

Arduino data saved to umap_data.h
Number of points: 64


# SPATIAL MAPPING


In [11]:
import numpy as np
from sklearn.cluster import KMeans

def generate_spatial_mapping(embeddings, n_clusters=5):
    """Map UMAP embeddings to LED grid preserving spatial relationships"""
    print("\nEmbeddings shape:", embeddings.shape)
    
    # First, identify clusters
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(embeddings)
    
    # Get cluster centers
    centers = kmeans.cluster_centers_
    
    # Create color map for clusters
    cluster_colors = [
        [255, 0, 0],    # Red
        [0, 255, 0],    # Green
        [0, 0, 255],    # Blue
        [255, 255, 0],  # Yellow
        [0, 255, 255]   # Cyan
    ]
    
    # Map to 8x8 grid
    x_min, x_max = embeddings[:, 0].min(), embeddings[:, 0].max()
    y_min, y_max = embeddings[:, 1].min(), embeddings[:, 1].max()
    
    # Generate Arduino code
    arduino_code = "const ColorPoint umapData[] = {\n"
    
    # Create grid mapping
    for i in range(min(64, len(embeddings))):
        # Map to LED grid (0-7 for both x and y)
        grid_x = int(((embeddings[i, 0] - x_min) / (x_max - x_min)) * 7)
        grid_y = int(((embeddings[i, 1] - y_min) / (y_max - y_min)) * 7)
        
        # Get cluster color
        cluster = clusters[i]
        color = cluster_colors[cluster]
        
        # Calculate distance to cluster center for brightness
        center = centers[cluster]
        distance = np.sqrt(np.sum((embeddings[i] - center) ** 2))
        max_distance = np.sqrt(np.sum((x_max - x_min) ** 2 + (y_max - y_min) ** 2))
        brightness = 1 - (distance / max_distance)  # Closer points are brighter
        
        # Apply brightness to color
        color = [int(c * brightness) for c in color]
        
        arduino_code += f"  {{{color[0]}, {color[1]}, {color[2]}, {grid_x}, {grid_y}}},"
        arduino_code += f"  // Point {i}, Cluster {cluster}\n"
    
    arduino_code += "};\n"
    return arduino_code

try:
    # Load the embeddings
    embeddings = np.load('umap_embeddings.npy')
    print("Loaded embeddings array with shape:", embeddings.shape)
    
    # Generate and save Arduino code
    arduino_code = generate_spatial_mapping(embeddings)
    
    # Save as header file
    with open("umap_data-SPATIAL.h", "w") as f:
        f.write("// Generated UMAP data for Arduino\n")
        f.write("#ifndef UMAP_DATA_H\n")
        f.write("#define UMAP_DATA_H\n\n")
        f.write("struct ColorPoint {\n")
        f.write("  uint8_t r;\n")
        f.write("  uint8_t g;\n")
        f.write("  uint8_t b;\n")
        f.write("  uint8_t x;\n")
        f.write("  uint8_t y;\n")
        f.write("};\n\n")
        f.write(arduino_code)
        f.write("\n#endif // UMAP_DATA_H\n")
    
    print(f"\nArduino data saved to umap_data.h")
    print(f"Number of points: {min(64, len(embeddings))}")

except Exception as e:
    print(f"Error: {str(e)}")
    print("\nPlease check your NPY file and try again.")

Loaded embeddings array with shape: (9882, 2)

Embeddings shape: (9882, 2)

Arduino data saved to umap_data.h
Number of points: 64


In [14]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def map_umap_to_grid(embeddings, pixels):
    """Map UMAP space directly to 8x8 LED grid"""
    print("\nCreating direct spatial mapping...")
    
    # Get UMAP space boundaries
    x_min, x_max = embeddings[:, 0].min(), embeddings[:, 0].max()
    y_min, y_max = embeddings[:, 1].min(), embeddings[:, 1].max()
    
    # Create 8x8 grid of sample points in UMAP space
    x_coords = np.linspace(x_min, x_max, 8)
    y_coords = np.linspace(y_min, y_max, 8)
    grid_points = []
    
    # Create grid coordinates
    for y in y_coords:
        for x in x_coords:
            grid_points.append([x, y])
    grid_points = np.array(grid_points)
    
    # Find nearest UMAP points for each grid position
    nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(embeddings)
    distances, indices = nbrs.kneighbors(grid_points)
    
    # Generate Arduino code
    arduino_code = "const ColorPoint umapData[] = {\n"
    
    for i in range(64):
        # Get color from nearest UMAP point
        color = pixels[indices[i][0]]
        grid_x = i % 8
        grid_y = i // 8
        
        arduino_code += f"  {{{int(color[0])}, {int(color[1])}, {int(color[2])}, {grid_x}, {grid_y}}},"
        arduino_code += f"  // LED {i}, UMAP point {indices[i][0]}\n"
    
    arduino_code += "};\n"
    
    # Print some stats
    print(f"UMAP space mapped to 8x8 grid:")
    print(f"X range: {x_min:.2f} to {x_max:.2f}")
    print(f"Y range: {y_min:.2f} to {y_max:.2f}")
    
    return arduino_code

try:
    # Load the embeddings and their original colors
    data = np.load('umap_embeddings.npy')  # Make sure to save both embeddings and pixels
    embeddings = data['embedding']
    pixels = data['pixels']
    
    # Generate and save Arduino code
    arduino_code = map_umap_to_grid(embeddings, pixels)
    
    with open("umap_data3.h", "w") as f:
        f.write("// Generated UMAP data for Arduino\n")
        f.write("#ifndef UMAP_DATA_H\n")
        f.write("#define UMAP_DATA_H\n\n")
        f.write("struct ColorPoint {\n")
        f.write("  uint8_t r;\n")
        f.write("  uint8_t g;\n")
        f.write("  uint8_t b;\n")
        f.write("  uint8_t x;\n")
        f.write("  uint8_t y;\n")
        f.write("};\n\n")
        f.write(arduino_code)
        f.write("\n#endif // UMAP_DATA_H\n")
    
    print(f"\nArduino data saved to umap_data.h")

except Exception as e:
    print(f"Error: {str(e)}")
    print("\nPlease check your NPY file and try again.")

Error: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

Please check your NPY file and try again.
